In [14]:
import spacy
from pydantic import BaseModel
from typing import List

In [2]:
nlp = spacy.load("en_core_web_sm")

In [22]:
import re
from typing import List, Dict, Any

def extract_scene_title(raw_text: str) -> str:
    """Extracts the scene title."""
    match = re.search(r"Scene Title:\s*\"(.*?)\"", raw_text)
    return match.group(1) if match else ""

def extract_section(raw_text: str, section_name: str) -> str:
    """Extracts specific sections of the raw text."""
    pattern = rf"{section_name}:\s*(.*?)(?=\n\n|\Z)"
    match = re.search(pattern, raw_text, re.DOTALL)
    return match.group(1).strip() if match else ""

def extract_characters(raw_text: str) -> List[Dict[str, Any]]:
    """Extracts characters and their attributes."""
    characters_text = extract_section(raw_text, "Characters")
    characters = []
    for block in re.split(r"\n\n", characters_text.strip()):
        lines = block.strip().split("\n")
        try:
            name, role = lines[0].split(" (")
            role = role.rstrip(")")
        except ValueError:
            continue  # Skip malformed entries
        details = {}
        for line in lines[1:]:
            if ": " in line:
                key, value = line.split(": ", maxsplit=1)
                details[key.lower()] = value
        characters.append({"name": name.strip(), "role": role.strip(), **details})
    return characters

def extract_dialogue(raw_text: str) -> List[Dict[str, str]]:
    """Extracts dialogue and speaker information."""
    dialogue_text = extract_section(raw_text, "Dialogue")
    dialogues = []
    for line in dialogue_text.split("\n"):
        if ": " in line:
            speaker, text = line.split(": ", maxsplit=1)
            tone = "shouting" if "!" in text else "smirking" if "smirk" in text else "neutral"
            dialogues.append({"speaker": speaker.strip(), "text": text.strip(), "tone": tone})
    return dialogues

def extract_actions(raw_text: str) -> List[Dict[str, str]]:
    """Extracts actions and their effects."""
    actions_text = extract_section(raw_text, "Action")
    actions = []
    for line in actions_text.split("\n"):
        if line.strip():  # Skip empty lines
            actions.append({"description": line.strip()})
    return actions


def extract_additional_notes(raw_text: str) -> List[str]:
    """Extracts additional notes."""
    notes_text = extract_section(raw_text, "Additional Notes")
    return [note.strip() for note in notes_text.split("\n")]

def process_raw_text(raw_text: str) -> Dict[str, Any]:
    """Processes the raw text and organizes it into a structured format."""
    return {
        "title": extract_scene_title(raw_text),
        "description": {
            "setting": extract_section(raw_text, "Setting"),
            "characters": extract_characters(raw_text),
            "dialogue": extract_dialogue(raw_text),
            "actions": extract_actions(raw_text),
            "background_details": extract_section(raw_text, "Background Details"),
            "additional_notes": extract_additional_notes(raw_text)
        }
    }

# Example usage
raw_text = '''
Scene Title: "The Rooftop Confrontation"

Scene Description:

Setting:

Time: Late evening, around sunset. The sky is painted in shades of orange, pink, and purple.
Location: The rooftop of a high school building. There are water tanks, a metal fence lining the edges, and a few discarded chairs near the entrance.
Atmosphere: The wind is strong, and papers are rustling around. A tense, almost electric mood is palpable.
Characters:

Protagonist (Hiro):
Appearance: A teenage boy, 16 years old, wearing a slightly torn school uniform. His hair is messy, and there’s a visible cut on his cheek.
Pose: Standing in a defensive stance, fists clenched, eyes blazing with determination.
Emotion: A mix of fear and courage as he faces his opponent.
Antagonist (Akira):
Appearance: A tall figure, 17 years old, with spiked hair and a leather jacket. He has a smirk and a wooden staff resting on his shoulder.
Pose: Leaning casually against the fence but with an intimidating aura.
Emotion: Confidence and a hint of malice.

Dialogue:

Hiro (shouting): “I won’t let you harm anyone else!”
Akira (smirking): “You? Stop me? Let’s see you try.”

Action:

Hiro steps forward, the wind causing his tie to flutter behind him.
Akira straightens up, spinning the wooden staff in his hand.
Background Details:

Behind Hiro: The door to the rooftop, slightly ajar, with light spilling out.
Behind Akira: The sun setting on the horizon, casting long shadows.

Additional Notes:

Include sound effects like "WHOOSH" for the wind and "CLANG" for the metallic noise as Hiro accidentally kicks a chair.
Focus on dramatic lighting, with the setting sun highlighting their silhouettes.
'''

# Process the text
processed_data = process_raw_text(raw_text)

# Print the structured data
from pprint import pprint
pprint(processed_data)


{'description': {'actions': [{'description': 'Hiro steps forward, the wind '
                                             'causing his tie to flutter '
                                             'behind him.'},
                             {'description': 'Akira straightens up, spinning '
                                             'the wooden staff in his hand.'},
                             {'description': 'Background Details:'}],
                 'additional_notes': ['Include sound effects like "WHOOSH" for '
                                      'the wind and "CLANG" for the metallic '
                                      'noise as Hiro accidentally kicks a '
                                      'chair.',
                                      'Focus on dramatic lighting, with the '
                                      'setting sun highlighting their '
                                      'silhouettes.'],
                 'background_details': 'Behind Hiro: The door to the r